In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

def run_era_scaled_pca(input_filename):
    # 1. --- LOAD DATABASE ---
    df = pd.read_csv(input_filename)

    # 2. --- CORE FEATURE ENGINEERING ---
    df['All_NBA_Teams'] = pd.to_numeric(df['All_NBA_Teams'], errors='coerce').fillna(0.0)
    df['Seasons_Played'] = pd.to_numeric(df['Seasons_Played'], errors='coerce').fillna(1.0)
    
    # Target feature protected from intra-era scaling adjustment
    protected_feature = 'All_NBA_per_Season'
    df[protected_feature] = df['All_NBA_Teams'] / df['Seasons_Played']

    # 3. --- SEPARATE AND CLEAN PCA FEATURES ---
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    ignore_cols = ['Modern_Consensus_Rank', 'Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank', 
                   'Optimized_Rank', 'Optimized_Score', 'New_Deviation', 'Sorted_Rank', 'Consensus_Rank']
    
    features_to_scale = [c for c in numeric_cols if c not in ignore_cols and c != protected_feature]

    # Handle missing values safely
    all_analysis_cols = features_to_scale + [protected_feature]
    for col in all_analysis_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median() if not df[col].isna().all() else 0.0)

    # 4. --- INTRA-ERA Z-SCORE STANDARD SCALING (WITH FLOAT CONVERSION) ---
    # FIX: Explicitly cast columns to float64 so they can receive decimal Z-scores without crashing
    df_era_scaled = df.copy()
    df_era_scaled[features_to_scale] = df_era_scaled[features_to_scale].astype(float)

    for era in df['Era_Label'].unique():
        era_mask = df['Era_Label'] == era
        if era_mask.sum() > 1:
            for col in features_to_scale:
                era_mean = df.loc[era_mask, col].mean()
                era_std = df.loc[era_mask, col].std()
                # Continuous Z-score assignment is safe now!
                df_era_scaled.loc[era_mask, col] = (df.loc[era_mask, col] - era_mean) / (era_std if era_std != 0 else 1.0)

    # 5. --- RE-CONSTRUCT PCA MATRIX ---
    # The protected feature bypasses the loop above and enters raw
    X_pca = df_era_scaled[all_analysis_cols].values

    # 6. --- EXECUTE PCA ANALYSIS ---
    n_components = 4
    pca = PCA(n_components=n_components)
    pca.fit(X_pca)

    # 7. --- EXTRACT FEATURE LOADINGS (ABSOLUTE MAGNITUDE IMPACT) ---
    loadings_df = pd.DataFrame(
        np.abs(pca.components_.T),
        index=all_analysis_cols,
        columns=[f'PC{i+1}_Importance' for i in range(n_components)]
    )

    # 8. --- DISPLAY INSIGHTS ---
    print("\n📊 PCA EXPLAINED VARIANCE RATIO BY COMPONENT:")
    for idx, var in enumerate(pca.explained_variance_ratio_):
        print(f" - Component {idx+1}: {var*100:.2f}%")
    print(f" 📈 Total System Variance Explained: {sum(pca.explained_variance_ratio_)*100:.2f}%\n")

    print("🔥 ================= TOP 10 ERA-NORMALIZED DRIVERS (PC1) ================= 🔥")
    print(loadings_df['PC1_Importance'].sort_values(ascending=False).head(10).to_string())
    
    print("\n🛡️ ================= PROTECTED FEATURE IMPORTANCE PROFILE ================= 🛡️")
    print(loadings_df.loc[protected_feature].to_string())

# --- EXECUTE THE BLOCK ---
if __name__ == "__main__":
    run_era_scaled_pca(os.path.join('..', 'nba_modern_filtered_master.csv'))

📖 Loaded '..\nba_modern_filtered_master.csv' with 88 players.

📊 PCA EXPLAINED VARIANCE RATIO BY COMPONENT:
 - Component 1: 54.10%
 - Component 2: 19.54%
 - Component 3: 5.59%
 - Component 4: 4.38%
 📈 Total System Variance Explained: 83.62%

🔥 ================= TOP 10 ERA-NORMALIZED DRIVERS (PC1) ================= 🔥
Playoff_WS_per_48           0.268420
Career_PER                  0.268312
Career_WS_per_48            0.268305
Playoff_PER                 0.268220
Peak_5Yr_PER                0.251492
Modern_Athletic_Raw_Rank    0.250394
Modern_BR_Raw_Rank          0.248297
All_NBA_PS                  0.245678
All_NBA_Teams               0.245047
Playoff_PPG                 0.237817

🛡️ ================= PROTECTED FEATURE IMPORTANCE PROFILE ================= 🛡️
PC1_Importance    0.059520
PC2_Importance    0.015502
PC3_Importance    0.034028
PC4_Importance    0.021362


In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score

def run_pca_driven_analytics_pipeline(input_filename, output_filename):
    # 1. --- LOAD DATABASE ---
    input_filename = os.path.join('..', 'nba_modern_filtered_master.csv')

    df = pd.read_csv(input_filename)
    print(f"📖 Loaded '{input_filename}' containing {len(df)} players.")

    # 2. --- CORE FEATURE ENGINEERING & PROTECTION ---
    df['All_NBA_Teams'] = pd.to_numeric(df['All_NBA_Teams'], errors='coerce').fillna(0.0)
    df['Seasons_Played'] = pd.to_numeric(df['Seasons_Played'], errors='coerce').fillna(1.0)
    
    protected_feature = 'All_NBA_per_Season'
    df[protected_feature] = df['All_NBA_Teams'] / df['Seasons_Played']

    # 3. --- ISOLATE NUMERIC PROFILE FEATURES ---
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    ignore_cols = ['Modern_Consensus_Rank', 'Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank', 
                   'Optimized_Rank', 'Optimized_Score', 'New_Deviation', 'Sorted_Rank', 'Consensus_Rank',
                   'Custom_Model_Rank', 'Custom_Index_Score', 'Rank_Deviation']
    
    features_to_scale = [c for c in numeric_cols if c not in ignore_cols and c != protected_feature]

    # Handle missing values across columns safely via the median
    all_analysis_cols = features_to_scale + [protected_feature]
    for col in all_analysis_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median() if not df[col].isna().all() else 0.0)

    # 4. --- INTRA-ERA Z-SCORE STANDARD SCALING ---
    df_era_scaled = df.copy()
    df_era_scaled[features_to_scale] = df_era_scaled[features_to_scale].astype(float)

    for era in df['Era_Label'].unique():
        era_mask = df['Era_Label'] == era
        if era_mask.sum() > 1:
            for col in features_to_scale:
                era_mean = df.loc[era_mask, col].mean()
                era_std = df.loc[era_mask, col].std()
                df_era_scaled.loc[era_mask, col] = (df.loc[era_mask, col] - era_mean) / (era_std if era_std != 0 else 1.0)

    # 5. --- RUN MATHEMATICAL PCA ---
    X_pca = df_era_scaled[all_analysis_cols].values
    pca = PCA(n_components=4) # Target the dominant variation axis
    
    # Map coordinates along the PC1 vector
    df['PCA_Index_Score'] = pca.fit_transform(X_pca)[:, 0]

    # 6. --- COMPUTE METRICS AGAINST MEDIA TARGETS ---
    if 'Modern_Consensus_Rank' not in df.columns:
        raw_cols = ['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']
        valid_raw = [c for c in raw_cols if c in df.columns]
        df['Modern_Consensus_Rank'] = df[valid_raw].mean(axis=1) if valid_raw else np.nan

    # Vector alignment: Ensure a higher score translates mathematically to an elite rank
    if df['PCA_Index_Score'].corr(df['Modern_Consensus_Rank']) > 0:
        df['PCA_Index_Score'] = -df['PCA_Index_Score']

    # Generate dense rankings from the PCA transformation
    df['PCA_Model_Rank'] = df['PCA_Index_Score'].rank(ascending=False, method='min').astype(int)
    
    df_eval = df.dropna(subset=['Modern_Consensus_Rank'])
    if not df_eval.empty:
        df['Rank_Deviation'] = df['PCA_Model_Rank'] - df['Modern_Consensus_Rank']
        mae = df['Rank_Deviation'].abs().mean()
        r2_ranking = r2_score(df_eval['Modern_Consensus_Rank'], df_eval['PCA_Model_Rank'])
    else:
        df['Rank_Deviation'] = 0.0
        mae, r2_ranking = 0.0, 0.0

    # Sort final dataframe by the data-driven ranking
    df_final = df.sort_values(by='PCA_Model_Rank')

    # 7. --- EXPORT FILE ---
    df_final.to_csv(output_filename, index=False)
    print(f"💾 Success! Exported pure PCA model results to '{output_filename}'")
    print(f"📊 Global Mean Absolute Error (MAE): {mae:.2f} spots")
    print(f"📉 Data-Driven Variance R² Score: {r2_ranking:.4f}\n")

    # 8. --- DISPLAY LEADERBOARD BREAKDOWN ---
    print("🔮 ==================== PURE PCA DRIVEN TOP 15 ==================== 🔮")
    print(f"{'PCA Rank':<10} | {'Player Name':<22} | {'Era':<12} | {'Consensus':<10}")
    print("-" * 65)
    
    for _, row in df_final.head(15).iterrows():
        print(f"{int(row['PCA_Model_Rank']):<10} | {row['Player_Name']:<22} | {row['Era_Label']:<12} | {row['Modern_Consensus_Rank']:<10.2f}")

# --- EXECUTE SCRIPT ---
if __name__ == "__main__":
    target_database = os.path.join('..', 'nba_modern_filtered_master.csv') 
    output_database = 'nba_pca_model_results.csv'
    
    run_pca_driven_analytics_pipeline(target_database, output_database)

📖 Loaded '..\nba_modern_filtered_master.csv' containing 88 players.
💾 Success! Exported pure PCA model results to 'nba_pca_model_results.csv'
📊 Global Mean Absolute Error (MAE): 9.91 spots
📉 Data-Driven Variance R² Score: 0.7381

🔮 ==================== PURE PCA DRIVEN TOP 15 ==================== 🔮
PCA Rank   | Player Name            | Era          | Consensus 
-----------------------------------------------------------------
1          | Michael Jordan         | 1990-1999    | 1.00      
2          | LeBron James           | 2010-2019    | 2.00      
3          | Kobe Bryant            | 2000-2009    | 7.67      
4          | Shaquille O'Neal       | 2000-2009    | 5.67      
5          | Kareem Abdul-Jabbar    | 1980-1989    | 3.00      
6          | Larry Bird             | 1980-1989    | 7.33      
7          | Tim Duncan             | 2000-2009    | 5.33      
8          | Karl Malone            | 1990-1999    | 14.67     
9          | Kevin Durant           | 2010-2019    | 10.67 